In [1]:
import random
import numpy as np
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [2]:
# =====================================================
# QUESTION 5 — LANGUAGE MODELING
# WikiText-2: N-gram vs LSTM Language Model
# Metric: Perplexity
# Text Generation: short samples
# =====================================================

# 1. Install libraries
!pip install -q "datasets<4.0.0" nltk

# =====================================================
# 2. Imports and seed
# =====================================================

import random
import math
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import nltk

from datasets import load_dataset
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

nltk.download("punkt")
nltk.download("punkt_tab")

# =====================================================
# 3. Load WikiText-2 dataset
# =====================================================

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

print("\nDataset structure:")
print(dataset)

print("\nExample text:")
print(dataset["train"][10]["text"])

# =====================================================
# 4. Preprocessing
# =====================================================

def tokenize_text(text):
    """
    Simple word-level tokenizer:
    - lowercase
    - remove empty lines
    - split punctuation
    - tokenize with nltk
    """
    text = text.lower().strip()

    if len(text) == 0:
        return []

    # Remove WikiText section titles like = heading =
    text = re.sub(r"=+", " ", text)

    tokens = nltk.word_tokenize(text)

    return tokens


def build_token_list(split, max_lines=None):
    all_tokens = []

    count = 0

    for example in dataset[split]:
        text = example["text"]
        tokens = tokenize_text(text)

        if len(tokens) > 0:
            all_tokens.extend(tokens)

        count += 1

        if max_lines is not None and count >= max_lines:
            break

    return all_tokens


# Use subsets for speed
train_tokens_raw = build_token_list("train", max_lines=8000)
valid_tokens_raw = build_token_list("validation", max_lines=1000)
test_tokens_raw = build_token_list("test", max_lines=1000)

print("\nRaw token counts:")
print("Train:", len(train_tokens_raw))
print("Validation:", len(valid_tokens_raw))
print("Test:", len(test_tokens_raw))

# =====================================================
# 5. Vocabulary
# =====================================================

MAX_VOCAB_SIZE = 10000
MIN_FREQ = 2

counter = Counter(train_tokens_raw)

special_tokens = ["<pad>", "<unk>", "<sos>", "<eos>"]

vocab = {tok: idx for idx, tok in enumerate(special_tokens)}

for word, freq in counter.most_common(MAX_VOCAB_SIZE - len(special_tokens)):
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

id2word = {idx: word for word, idx in vocab.items()}

PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]
SOS_IDX = vocab["<sos>"]
EOS_IDX = vocab["<eos>"]

print("\nVocabulary size:", len(vocab))


def numericalize(tokens):
    return [vocab.get(tok, UNK_IDX) for tok in tokens]


train_ids = numericalize(train_tokens_raw)
valid_ids = numericalize(valid_tokens_raw)
test_ids = numericalize(test_tokens_raw)

print("\nNumericalized example:")
print(train_tokens_raw[:20])
print(train_ids[:20])

# =====================================================
# PART A — TRIGRAM LANGUAGE MODEL
# =====================================================

# Add sentence boundary tokens for n-gram modeling
train_tokens_ngram = ["<sos>", "<sos>"] + [
    tok if tok in vocab else "<unk>" for tok in train_tokens_raw
] + ["<eos>"]

valid_tokens_ngram = ["<sos>", "<sos>"] + [
    tok if tok in vocab else "<unk>" for tok in valid_tokens_raw
] + ["<eos>"]

test_tokens_ngram = ["<sos>", "<sos>"] + [
    tok if tok in vocab else "<unk>" for tok in test_tokens_raw
] + ["<eos>"]


def train_trigram_model(tokens):
    trigram_counts = defaultdict(Counter)
    bigram_counts = Counter()

    for i in range(2, len(tokens)):
        context = (tokens[i - 2], tokens[i - 1])
        word = tokens[i]

        trigram_counts[context][word] += 1
        bigram_counts[context] += 1

    return trigram_counts, bigram_counts


trigram_counts, bigram_counts = train_trigram_model(train_tokens_ngram)

VOCAB_SIZE = len(vocab)
ALPHA = 0.1


def trigram_probability(word, context):
    """
    Add-k smoothing:
    P(w | w_{i-2}, w_{i-1}) =
    (count(context, word) + alpha) / (count(context) + alpha * |V|)
    """
    count_context_word = trigram_counts[context][word]
    count_context = bigram_counts[context]

    probability = (count_context_word + ALPHA) / (count_context + ALPHA * VOCAB_SIZE)

    return probability


def trigram_perplexity(tokens):
    log_prob_sum = 0.0
    n = 0

    for i in range(2, len(tokens)):
        context = (tokens[i - 2], tokens[i - 1])
        word = tokens[i]

        prob = trigram_probability(word, context)

        log_prob_sum += math.log(prob)
        n += 1

    avg_neg_log_prob = -log_prob_sum / n
    ppl = math.exp(avg_neg_log_prob)

    return ppl


trigram_valid_ppl = trigram_perplexity(valid_tokens_ngram)
trigram_test_ppl = trigram_perplexity(test_tokens_ngram)

print("\n=========================")
print("TRIGRAM LANGUAGE MODEL")
print("=========================")
print("Validation Perplexity:", trigram_valid_ppl)
print("Test Perplexity:", trigram_test_ppl)


def generate_trigram_text(start_words=None, max_len=30):
    if start_words is None:
        generated = ["<sos>", "<sos>"]
    else:
        generated = ["<sos>"] + start_words.lower().split()

        if len(generated) < 2:
            generated = ["<sos>"] + generated

    for _ in range(max_len):
        context = (generated[-2], generated[-1])

        candidates = list(vocab.keys())

        probs = np.array([
            trigram_probability(word, context)
            for word in candidates
        ])

        probs = probs / probs.sum()

        next_word = np.random.choice(candidates, p=probs)

        if next_word == "<eos>":
            break

        generated.append(next_word)

    output = [
        tok for tok in generated
        if tok not in ["<sos>", "<eos>", "<pad>"]
    ]

    return " ".join(output)


print("\nTrigram generated samples:")
for prompt in ["the", "in the", "he"]:
    print("Prompt:", prompt)
    print(generate_trigram_text(prompt, max_len=30))
    print()

# =====================================================
# PART B — LSTM LANGUAGE MODEL
# =====================================================

SEQ_LEN = 35
BATCH_SIZE = 64


class LanguageModelingDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        self.token_ids = token_ids
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.token_ids) - self.seq_len)

    def __getitem__(self, idx):
        x = self.token_ids[idx:idx + self.seq_len]
        y = self.token_ids[idx + 1:idx + self.seq_len + 1]

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


train_lm_dataset = LanguageModelingDataset(train_ids, SEQ_LEN)
valid_lm_dataset = LanguageModelingDataset(valid_ids, SEQ_LEN)
test_lm_dataset = LanguageModelingDataset(test_ids, SEQ_LEN)

train_lm_loader = DataLoader(train_lm_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_lm_loader = DataLoader(valid_lm_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_lm_loader = DataLoader(test_lm_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("\nLSTM dataset sizes:")
print("Train batches:", len(train_lm_loader))
print("Validation batches:", len(valid_lm_loader))
print("Test batches:", len(test_lm_loader))


class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids, hidden=None):
        embedded = self.dropout(self.embedding(input_ids))

        outputs, hidden = self.lstm(embedded, hidden)

        outputs = self.dropout(outputs)

        logits = self.fc(outputs)

        return logits, hidden


VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3

lstm_model = LSTMLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

print("\nLSTM model created:")
print(lstm_model)


def train_lstm_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits, _ = model(x)

        vocab_size = logits.shape[-1]

        loss = criterion(
            logits.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


def evaluate_lstm(model, dataloader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits, _ = model(x)

            vocab_size = logits.shape[-1]

            loss = criterion(
                logits.reshape(-1, vocab_size),
                y.reshape(-1)
            )

            total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


N_EPOCHS = 5

print("\nTraining LSTM Language Model...")

best_valid_ppl = float("inf")
best_epoch = 0

for epoch in range(N_EPOCHS):
    train_loss, train_ppl = train_lstm_epoch(
        lstm_model,
        train_lm_loader,
        optimizer,
        criterion
    )

    valid_loss, valid_ppl = evaluate_lstm(
        lstm_model,
        valid_lm_loader,
        criterion
    )

    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train PPL: {train_ppl:.2f}")
    print(f"Validation Loss: {valid_loss:.4f} | Validation PPL: {valid_ppl:.2f}")
    print("-" * 50)

    if valid_ppl < best_valid_ppl:
        best_valid_ppl = valid_ppl
        best_epoch = epoch + 1
        torch.save(lstm_model.state_dict(), "best_lstm_lm.pt")

# Load best model
lstm_model.load_state_dict(torch.load("best_lstm_lm.pt"))

test_loss, lstm_test_ppl = evaluate_lstm(
    lstm_model,
    test_lm_loader,
    criterion
)

print("\n=========================")
print("LSTM LANGUAGE MODEL")
print("=========================")
print("Best Epoch:", best_epoch)
print("Best Validation Perplexity:", best_valid_ppl)
print("Test Perplexity:", lstm_test_ppl)

# =====================================================
#  Generation with LSTM
# =====================================================

def generate_lstm_text(model, prompt="the", max_len=30, temperature=1.0):
    model.eval()

    tokens = tokenize_text(prompt)
    ids = [vocab.get(tok, UNK_IDX) for tok in tokens]

    if len(ids) == 0:
        ids = [SOS_IDX]

    input_ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

    generated = ids.copy()

    hidden = None

    with torch.no_grad():
        for _ in range(max_len):
            logits, hidden = model(input_ids, hidden)

            next_token_logits = logits[:, -1, :] / temperature

            probs = torch.softmax(next_token_logits, dim=-1)

            next_id = torch.multinomial(probs, num_samples=1).item()

            if next_id == EOS_IDX:
                break

            generated.append(next_id)

            input_ids = torch.tensor([[next_id]], dtype=torch.long).to(device)

    words = [
        id2word.get(idx, "<unk>")
        for idx in generated
        if id2word.get(idx, "<unk>") not in ["<pad>", "<sos>", "<eos>"]
    ]

    return " ".join(words)


print("\nLSTM generated samples:")
for prompt in ["the", "in the", "he"]:
    print("Prompt:", prompt)
    print(generate_lstm_text(lstm_model, prompt=prompt, max_len=30, temperature=0.8))
    print()

# =====================================================
# Final Results Table
# =====================================================

results_q5 = {
    "Model": ["Trigram N-gram", "LSTM"],
    "Validation Perplexity": [trigram_valid_ppl, best_valid_ppl],
    "Test Perplexity": [trigram_test_ppl, lstm_test_ppl]
}

results_df_q5 = pd.DataFrame(results_q5)

print("\n=========================")
print("FINAL Q5 RESULTS TABLE")
print("=========================")
print(results_df_q5)

# Save outputs
results_df_q5.to_csv("q5_language_modeling_results.csv", index=False)

sample_outputs = {
    "Prompt": ["the", "in the", "he"],
    "Trigram Output": [
        generate_trigram_text("the", max_len=30),
        generate_trigram_text("in the", max_len=30),
        generate_trigram_text("he", max_len=30)
    ],
    "LSTM Output": [
        generate_lstm_text(lstm_model, prompt="the", max_len=30, temperature=0.8),
        generate_lstm_text(lstm_model, prompt="in the", max_len=30, temperature=0.8),
        generate_lstm_text(lstm_model, prompt="he", max_len=30, temperature=0.8)
    ]
}

samples_df_q5 = pd.DataFrame(sample_outputs)
samples_df_q5.to_csv("q5_generated_samples.csv", index=False)

print("\nGenerated sample table:")
print(samples_df_q5)

print("\nSaved files:")
print("q5_language_modeling_results.csv")
print("q5_generated_samples.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 23.2 MB/s eta 0:00:00
Device: cuda


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]


Dataset structure:
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Example text:
 The game 's battle system , the BliTZ system , is carried over directly from Valkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters ' turns . Each character has a field and distance of movement limited by their Action Gauge . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting lo